# Embedding Experiments

このノートブックでは、catchup-aiのEmbedding機能を実験します。

## 内容
1. 環境セットアップ
2. OpenAI Embeddingの生成
3. コサイン類似度の計算
4. 類似テキスト検索のデモ
5. (オプション) Voyage AIとの比較

## 1. 環境セットアップ

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

In [ ]:
from dotenv import load_dotenv

env_path = project_root / ".env"
load_dotenv(env_path)
print(f"Loaded .env from: {env_path}")

In [ ]:
from catchup_ai.core.embedding import (
    create_embedding_service,
    EmbeddingResult,
    ArticleEmbeddingInput,
)
from catchup_ai.infra.config.settings import get_settings

settings = get_settings()
print(f"Environment: {settings.environment}")
print(f"Embedding Provider: {settings.embedding.provider.value}")

## 2. Embedding生成

In [ ]:
embedding_service = create_embedding_service()
print(f"Created: {type(embedding_service).__name__}")

In [ ]:
text = "Pythonは人気のあるプログラミング言語です。"
result = embedding_service.embed_text(text)

print(f"Text: {text}")
print(f"Model: {result.model}")
print(f"Provider: {result.provider}")
print(f"Dimension: {result.dimension}")
print(f"Vector (first 5): {result.vector[:5]}")

## 3. コサイン類似度

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    a = np.array(vec1)
    b = np.array(vec2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
texts = [
    "Pythonは人気のあるプログラミング言語です。",
    "Pythonはデータサイエンスでよく使われる言語です。",
    "JavaScriptはウェブ開発で使われる言語です。",
    "今日の天気は晴れです。",
]

results = embedding_service.embed_texts(texts)

n = len(results)
for i in range(n):
    for j in range(i+1, n):
        sim = cosine_similarity(results[i].vector, results[j].vector)
        print(f"[{i}] vs [{j}]: {sim:.3f}")
        print(f"  {texts[i][:30]}...")
        print(f"  {texts[j][:30]}...")
        print()

## 4. 類似記事検索

In [ ]:
articles = [
    {"id": 1, "title": "Python 3.13の新機能", "content": "Python 3.13では多くの改善があります。"},
    {"id": 2, "title": "機械学習入門", "content": "Pythonで機械学習を始めましょう。"},
    {"id": 3, "title": "Rustで高速CLIツール", "content": "Rustで安全なツールを作る。"},
    {"id": 4, "title": "React開発", "content": "TypeScriptでモダンなフロントエンド。"},
    {"id": 5, "title": "RAGチャットボット", "content": "独自データで回答を生成。"},
]

article_inputs = [
    ArticleEmbeddingInput(article_id=a["id"], title=a["title"], content=a["content"])
    for a in articles
]
article_embeddings = embedding_service.embed_articles(article_inputs)
print(f"Generated {len(article_embeddings)} embeddings")

In [ ]:
def search(query, top_k=3):
    query_result = embedding_service.embed_text(query)
    sims = [(a, cosine_similarity(query_result.vector, e.vector)) 
            for a, e in zip(articles, article_embeddings)]
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:top_k]

queries = ["Pythonでデータ分析", "AIチャットボット", "Webフロントエンド"]
for q in queries:
    print(f"Query: {q}")
    for a, sim in search(q, 2):
        print(f"  [{sim:.3f}] {a['title']}")
    print()

## まとめ

- `create_embedding_service()`でプロバイダーに応じたサービスを作成
- コサイン類似度でベクトル間の類似性を測定
- クエリをEmbeddingして記事検索が可能